Imports

In [ ]:
import pandas as pd
import numpy as np

Aggregate AVM to ZIP level

In [ ]:
avm = pd.read_csv("AVM.csv")
mig = pd.read_csv("OC2025_ZIPMigration.csv")

avm_zip = (
    avm.groupby("ZipCode")
    .agg(
        median_value   = ("FinalValue",  "median"),
        median_high    = ("HighValue",   "median"),
        median_low     = ("LowValue",    "median"),
        property_count = ("RecordId",    "count"),
    )
    .reset_index()
) 

AVM Features - How wide the AVM confidence band is relative to value (A wide band = model uncertainty = price instability in that ZIP)

In [ ]:
avm_zip["price_spread_pct"] = (
    (avm_zip["median_high"] - avm_zip["median_low"])
    / avm_zip["median_value"]
)

Migration Features (Positive = more ppl leaving than entering)

In [ ]:
mig["pct_leave"] = mig["PctLeave"] / 100.0

mig["net_migration_rate"] = (
    (mig["MovesOutOfZip"] - mig["MovesIntoZip"]) / mig["TotalMoves"]
)

Merge

In [ ]:
df = mig[
    ["ZipCode", "State", "ZipCode_Latitude", "ZipCode_Longitude",
     "MovesOutOfZip", "MovesIntoZip", "TotalMoves",
     "pct_leave", "net_migration_rate"]
].merge(
    avm_zip[["ZipCode", "median_value", "property_count", "price_spread_pct"]],
    on="ZipCode",
    how="left",   # some may lack AVM data
)

df["turnover_rate"] = df["TotalMoves"] / df["property_count"] # total moves relative to property stock

Min–max normalise each feature to 0 or 1

In [ ]:
FEATURES = ["pct_leave", "net_migration_rate", "price_spread_pct", "turnover_rate"]
 
def minmax(series: pd.Series) -> pd.Series:
    lo, hi = series.min(), series.max()
    if hi == lo:
        return pd.Series(0.5, index=series.index)   # all identical = neutral
    return (series - lo) / (hi - lo)
 
df_norm = df.copy()
for feat in FEATURES:
    df_norm[f"{feat}_norm"] = minmax(df[feat])

Composite Score

In [ ]:
WEIGHTS = {
    "pct_leave":          0.40,
    "net_migration_rate": 0.30,
    "price_spread_pct":   0.15,
    "turnover_rate":      0.15,
}
assert abs(sum(WEIGHTS.values()) - 1.0) < 1e-9, "Weights must sum to 1"
 
df_norm["flight_risk_score"] = sum(
    WEIGHTS[f] * df_norm[f"{f}_norm"] for f in FEATURES
)

Labeling Risk Tiers

In [ ]:
def risk_tier(score: float) -> str:
    if score >= 0.67:  return "High"
    if score >= 0.33:  return "Medium"
    return "Low"
 
df_norm["risk_tier"] = df_norm["flight_risk_score"].apply(risk_tier)

Results

In [ ]:
DISPLAY_COLS = [
    "ZipCode", "flight_risk_score", "risk_tier",
    "pct_leave", "net_migration_rate", "price_spread_pct", "turnover_rate",
    "median_value", "property_count",
    "ZipCode_Latitude", "ZipCode_Longitude",
]
 
results = (
    df_norm[DISPLAY_COLS]
    .sort_values("flight_risk_score", ascending=False)
    .reset_index(drop=True)
)

results.to_csv("zip_flight_risk_scores.csv", index=False)